# IBVAP — GPU Video Analytics Pipeline (Kaggle)

Drop videos into `input_videos/`, run all cells, collect annotated videos + full telemetry from `runs/`.

**Enable the GPU accelerator first:** Settings → Accelerator → GPU T4 x2 (or P100).

## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 2. Add your videos
Kaggle notebooks can read datasets directly from `/kaggle/input/`, or you can add a notebook-scratch folder.
Point `--input` at whichever holds your videos.

In [ ]:
import os, glob
# List any attached datasets
for p in sorted(glob.glob('/kaggle/input/*')):
    if os.path.isdir(p):
        vids = [f for f in glob.glob(p + '/**/*', recursive=True)
                if os.path.splitext(f)[1].lower() in ('.mp4','.avi','.mov','.mkv','.webm','.ts')]
        if vids:
            print(p, '->', len(vids), 'videos')

INPUT = '/kaggle/input'          # change if your videos live elsewhere
os.makedirs('input_videos', exist_ok=True)
print('\nworking dir:', os.getcwd())

## 3. Install dependencies

In [ ]:
!pip install -q ultralytics opencv-python-headless numpy pynvml psutil
import ultralytics, cv2
print('ultralytics', ultralytics.__version__, '| opencv', cv2.__version__)

## 4. Add the pipeline source
Add this repo as a Kaggle **Notebook** or **Dataset** (mount it under `/kaggle/input/`), then copy `ibvap/` + `run_ibvap.py` into the working directory.

In [ ]:
import glob, os, shutil
src = None
for cand in ['/kaggle/input/ibvap-src', '/kaggle/working/ibvap-src'] + glob.glob('/kaggle/input/*ibvap*'):
    if os.path.isdir(os.path.join(cand, 'ibvap')):
        src = cand
        break
if src is None:
    # Fall back: look for the package anywhere under /kaggle/input
    hits = [p for p in glob.glob('/kaggle/input/**/ibvap', recursive=True) if os.path.isdir(p)]
    src = os.path.dirname(hits[0]) if hits else None
print('source dir:', src)
if src:
    for item in ['ibvap', 'run_ibvap.py', 'requirements.txt']:
        s = os.path.join(src, item)
        if os.path.exists(s):
            if os.path.isdir(s):
                shutil.copytree(s, os.path.join('/kaggle/working', item), dirs_exist_ok=True)
            else:
                shutil.copy(s, '/kaggle/working/')
print('ready:', os.path.isdir('ibvap'), os.path.isfile('run_ibvap.py'))

## 5. (Optional) camera calibration + fence zones

In [ ]:
import json
calib = {
  "c01": {"mode": "ground", "horizon_y": 42, "cam_height_m": 6.0, "focal_px": 1180, "rig_m": 1.2},
  "c02": {"mode": "ground", "horizon_y": 45, "cam_height_m": 5.0, "focal_px": 940,  "rig_m": 0.8},
  "c03": {"mode": "aerial", "fixed_depth_m": 120},
  "c04": {"mode": "ground", "horizon_y": 48, "cam_height_m": 4.0, "focal_px": 1620, "rig_m": 0.6},
  "c11": {"mode": "sky"}
}
json.dump({"camera_calib": calib}, open('calib.json', 'w'), indent=2)

zones = {
  "c01": [{"id": "VF-02", "name": "VF-02 Fence North", "kind": "tripwire",
           "points": [[0.08, 0.72], [0.94, 0.55]], "direction": "both", "min_size_pct": 3}],
  "c06": [{"id": "VF-04", "name": "VF-04 Treeline", "kind": "polygon",
           "points": [[0.16, 0.40], [0.58, 0.32], [0.86, 0.58], [0.40, 0.74]], "min_size_pct": 3}]
}
json.dump(zones, open('zones.json', 'w'), indent=2)
print('calib.json + zones.json written')

## 6. Run the pipeline
`INPUT` points at wherever your videos are. `runs/` lands in `/kaggle/working` which is downloadable as notebook output.

In [ ]:
!python run_ibvap.py \
  --input {INPUT} \
  --output runs \
  --config calib.json \
  --zones zones.json \
  --device 0 \
  --imgsz 640 \
  --conf 0.25 \
  --run-name kaggle-t4

## 7. Read the telemetry

In [ ]:
import json, glob, os
run = sorted(glob.glob('runs/*'))[-1]
print('run dir:', run)
s = json.load(open(os.path.join(run, 'summary.json')))
print(json.dumps({
  'wall_clock': s['wall_clock'],
  'frames': s['frames'],
  'throughput': s['throughput'],
  'gpu_util': s['gpu']['utilisation_pct'],
  'errors': s['errors'],
}, indent=2))
print('\nstage share:')
for k, v in sorted(s['stage_share_pct'].items(), key=lambda x: -x[1]):
    print(f"  {k:24s} {v:6.2f}%  p95={s['stages'][k]['p95_ms']:8.2f}ms  calls={s['stages'][k]['calls']}")
print('\nfiles:')
for f in sorted(os.listdir(run)):
    print('  ', f, os.path.getsize(os.path.join(run, f)) if os.path.isfile(os.path.join(run, f)) else '<dir>')

## 8. Output
Everything under `/kaggle/working/runs/` is saved as notebook output and can be downloaded from the notebook viewer, or commit the notebook to produce a dataset version.

In [ ]:
import glob, os
run = sorted(glob.glob('runs/*'))[-1]
for f in sorted(glob.glob(os.path.join(run, 'videos', '*.mp4'))):
    print(f, f'{os.path.getsize(f)/1024/1024:.1f} MB')
print('\nsummary:', os.path.join(run, 'summary.json'))

## Notes

- Kaggle GPU sessions have a **9-hour limit** — chunk long batches.
- **`--max-frames 300`** for a smoke test.
- **`--no-video`** if you only want telemetry.
- `/kaggle/working` persists only for the session — commit the notebook to keep output.